# KETGM — Knowledge-Enhanced and Topic-Guided Domain Adaptation for ABSA

Full pipeline implementation of **Zeng et al. (IEEE TAFFC 2024)**.

| Step | Script | Paper Section |
|------|--------|---------------|
| 0 | Setup & Install | — |
| 1 | `data_loader.py` | — |
| 2 | `topic_extraction.py` | III-B1 |
| 3 | `knowledge_graph.py` | III-B1 |
| 4 | `pretrain_rgcn.py` | III-B2 |
| 5 | `pretrain_bert.py` | III-D |
| 6 | `train.py` | III-E / III-F |
| 7 | `evaluate.py` / `inference.py` | IV |

---
## 0 · Setup & Install

In [1]:
!pip install -q torch transformers spacy gensim scikit-learn seqeval tqdm lxml

In [2]:
!pip install -r requirements.txt

In [3]:
!python -m spacy download en_core_web_sm -q

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [4]:
import spacy
spacy.load('en_core_web_sm')

In [5]:
import os, sys, torch, warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from config import Config
from utils import set_seed

# Patch paths for Colab / Lightning
Config.PROJECT_ROOT   = PROJECT_ROOT
Config.DATA_DIR       = os.path.join(PROJECT_ROOT, 'data')
Config.RAW_DIR        = os.path.join(Config.DATA_DIR, 'raw')
Config.PROCESSED_DIR  = os.path.join(Config.DATA_DIR, 'processed')
Config.CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, 'checkpoints')
Config.CONCEPTNET_CSV_GZ = os.path.join(Config.DATA_DIR, 'conceptnet-assertions-5.7.0.csv.gz')
Config.CONCEPTNET_EN_PKL = os.path.join(Config.DATA_DIR, 'conceptnet_en.pkl')
Config.DOMAIN_CSV = {
    'restaurant': os.path.join(Config.DATA_DIR, 'restaurant.csv'),
    'laptop':     os.path.join(Config.DATA_DIR, 'laptop.csv'),
    'device':     os.path.join(Config.DATA_DIR, 'device.csv'),
    'service':    os.path.join(Config.DATA_DIR, 'service.csv'),
}
Config.DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
Config.ensure_dirs()
set_seed(Config.SEED)
DEVICE = Config.DEVICE

print(f'PyTorch {torch.__version__}')
print(f'CUDA    {torch.cuda.is_available()}  —  '
      f'{torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'Device  : {DEVICE}')
print(f'Project : {PROJECT_ROOT}')

PyTorch 2.8.0+cu128
CUDA    True  —  Tesla T4
Device  : cuda
Project : /teamspace/studios/this_studio


In [6]:
# ── Choose cross-domain task ──────────────────────────────────────
# Paper tasks: S→R, L→R, D→R, R→S, L→S, D→S, R→L, S→L, R→D, S→D
Config.SOURCE_DOMAIN = 'restaurant'  # train on this (labeled)
Config.TARGET_DOMAIN = 'laptop'      # test on this (unlabeled)
print(f'Cross-domain task: {Config.SOURCE_DOMAIN} → {Config.TARGET_DOMAIN}')

Cross-domain task: restaurant → laptop


In [7]:
# Download ConceptNet (run once — ~530 MB)
from utils import download_file
print('=== ConceptNet 5.7.0 CSV ===')
download_file(Config.CONCEPTNET_URL, Config.CONCEPTNET_CSV_GZ,
              'conceptnet-assertions-5.7.0.csv.gz')

=== ConceptNet 5.7.0 CSV ===
  ↓ Downloading conceptnet-assertions-5.7.0.csv.gz …


conceptnet-assertions-5.7.0.csv.gz: 498MB [00:05, 98.1MB/s]                              

  ✓ Saved to /teamspace/studios/this_studio/data/conceptnet-assertions-5.7.0.csv.gz


---
## 1 · Load Data (Source + Target CSVs)

In [8]:
import data_loader

data = data_loader.run(config=Config)
samples        = data['samples']
train_samples  = data['train_samples']
eval_samples   = data['eval_samples']
target_samples = data['target_samples']
pos2id         = data['pos2id']
dep2id         = data['dep2id']
nlp            = data['nlp']

Source domain: restaurant
Target domain: laptop
Parsed 3497 source sentences from restaurant.csv  (19.4s)
Parsed 1859 target sentences from laptop.csv  (10.0s)
Train: 2797   Eval: 700
POS tags: 48   DEP tags: 45

Verification — "Finally a curry that I can eat, enjoy and not suffer from gastritis from 3 hours..."
  Finally                O
  a                      O
  curry                → B-POS
  that                   O
  I                      O
  can                    O
  eat                    O
  ,                      O
  enjoy                  O
  and                    O
  not                    O
  suffer                 O
  from                   O
  gastritis              O
  from                   O


---
## 2 · Extract LDA Topics (TWLDA)

In [9]:
import topic_extraction

topic_words = topic_extraction.run(train_samples, target_samples, config=Config)

Extracted 115 topic words  (35.8s)
  Topic 0: ['garden', 'curry', 'hours', 'hard', 'sharp', 'hrs', 'set', 'fried', 'send', 'tuna']
  Topic 1: ['mouse', 'light', 'games', 'quick', 'fresh', 'ram', 'book', 'slow', 'toshiba', 'easy']
  Topic 2: ['cool', 'size', 'given', 'price', 'motherboard', 'problem', 'web', 'minutes', 'pizza', 'friendly']
  Topic 3: ['away', 'times', 'dinner', 'recommend', 'warranty', 'ambiance', 'sauce', 'extended', 'food', 'windows']
  Topic 4: ['longer', 'warrenty', 'nice', 'asked', 'wait', 'space', 'track', 'vista', 'prompt', 'fish']
  Topic 5: ['crust', 'prices', 'pad', 'seated', 'tried', 'external', 'rude', 'keys', 'dishes', 'staff']
  Topic 6: ['won', 'shrimp', 'ports', 'touch', 'table', 'told', 'long', 'best', 'drive', 'use']
  Topic 7: ['small', 'runs', 'thing', 'poor', 'battery', 'sent', 'great', 'installed', 'lunch', 'feel']
  Topic 8: ['indian', 'love', 'bit', 'good', 'wine', 'selection', 'absolutely', 'horrible', 'eaten', 'life']
  Topic 9: ['attentive', '

---
## 3 · Build Knowledge Graph from ConceptNet

In [10]:
import knowledge_graph

kg_result = knowledge_graph.run(samples, target_samples, topic_words, nlp, config=Config)
kg = kg_result['kg']

  ⏳ Streaming ConceptNet CSV (English only) …


ConceptNet: 34074917 edges [01:32, 366909.16 edges/s]


  ✓ Kept 3,423,004 English edges, 1,165,190 concepts
English ConceptNet loaded  (100.2s)
Concepts in adjacency list: 1,165,190
Seed tokens: 4631


KG paths: 100%|██████████| 4631/4631 [00:03<00:00, 1291.40 tokens/s]


  ✓ KG: 8,722 nodes, 22,068 edges, 37 relation types
  ✓ 3855/4631 seed tokens linked to topics
KG built in 3.6s
  Nodes : 8,722
  Edges : 22,068
  Rels  : 37
  Token→node mappings : 3,855
  Topic nodes in KG   : 113


---
## 4 · Pre-train R-GCN Autoencoder (400 epochs)

In [11]:
import pretrain_rgcn

rgcn_result  = pretrain_rgcn.run(kg, DEVICE, config=Config)
rgcn_model   = rgcn_result['rgcn_model']
all_tc       = rgcn_result['all_tc']
topic_tc     = rgcn_result['topic_tc']

R-GCN setup: 8,722 nodes · 37 rels · 22,068 edges
R-GCN params: 4,314,820


R-GCN:   0%|          | 0/400 [00:00<?, ?it/s]


✓ R-GCN pre-training done  (31.1s)
Node embeddings : torch.Size([8722, 200])
Topic embeddings: torch.Size([113, 200])


---
## 5 · Pre-train BERT with Syntactic Tasks (POS + DEP)

In [12]:
import pretrain_bert

bert_result = pretrain_bert.run(train_samples, eval_samples, pos2id, dep2id,
                                DEVICE, config=Config)
bert_syn = bert_result['bert_syn']

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

...

  BERT-Syn Epoch 3/5  loss=0.9642
    ★ New best


BERT-Syn Ep 04:   0%|          | 0/88 [00:00<?, ?it/s]

  BERT-Syn Epoch 4/5  loss=0.7165
    ★ New best


BERT-Syn Ep 05:   0%|          | 0/88 [00:00<?, ?it/s]

  BERT-Syn Epoch 5/5  loss=0.5744
    ★ New best

✓ BERT syntactic pre-training done  (379.3s)


---
## 6 · Train Full KETGM Model

In [13]:
import train

train_result = train.run(
    train_samples, eval_samples, pos2id, dep2id,
    kg, all_tc, topic_tc, DEVICE,
    bert_state_dict=bert_syn.state_dict(),
    rgcn_feat_map_state=rgcn_model.feat_map.state_dict(),
    rgcn_feat_recon_state=rgcn_model.feat_recon.state_dict(),
    config=Config,
)
model   = train_result['model']
best_f1 = train_result['best_f1']

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✓ Loaded feat_map weights from pre-trained R-GCN
  ✓ Loaded feat_recon weights from pre-trained R-GCN
  ✓ Loaded BERT weights from syntactic pre-training
KETGM params: 109,642,340
Batches/epoch: 88


Ep 01:   0%|          | 0/88 [00:00<?, ?it/s]

  Epoch  1/30  loss=1.4074  Micro-F1=0.4020
    ★ New best


...

  Epoch 28/30  loss=0.8435  Micro-F1=0.6659


Ep 29:   0%|          | 0/88 [00:00<?, ?it/s]

  Epoch 29/30  loss=0.8427  Micro-F1=0.6672


Ep 30:   0%|          | 0/88 [00:00<?, ?it/s]

  Epoch 30/30  loss=0.8420  Micro-F1=0.6448

✓ Training done  (2387s)  Best Micro-F1 = 0.6750


---
## 7 · Evaluate & Inference

In [14]:
import evaluate

eval_result = evaluate.run(
    eval_samples, pos2id, dep2id,
    kg, all_tc, topic_tc, DEVICE,
    model=model, config=Config,
)


═══ Evaluation Results ═══
Micro-F1: 0.6448
              precision    recall  f1-score   support

         NEG       0.52      0.63      0.57       264
         NEU       0.34      0.44      0.39       181
         POS       0.71      0.77      0.74       770

   micro avg       0.61      0.69      0.64      1215
   macro avg       0.53      0.61      0.56      1215
weighted avg       0.62      0.69      0.65      1215



In [15]:
import torch
import os

# Define a directory to save the model
save_dir = "saved_models"
os.makedirs(save_dir, exist_ok=True)

# Save the model's state_dict
model_path = os.path.join(save_dir, "KETGM.pth")
torch.save(model.state_dict(), model_path)

print(f"Model saved successfully at: {model_path}")
print(f"Best F1 score during training: {best_f1}")

Model saved successfully at: saved_models/KETGM.pth
Best F1 score during training: 0.6750303520841764


In [16]:
import inference

inference.run(
    model, eval_samples, pos2id, dep2id,
    kg, all_tc, topic_tc, DEVICE,
    num_display=5, config=Config,
)


Sentence:
They have great rolls , the triple color and norwegetan rolls , are awesome and filling .
Predictions:
They            -> O
have            -> O
great           -> O
rolls           -> B-POS
,               -> O
the             -> O
triple          -> B-POS
color           -> I-POS
and             -> I-POS
norwegetan      -> B-POS
rolls           -> I-POS
,               -> O
are             -> O
awesome         -> O
and             -> O
filling         -> O
.               -> O

Sentence:
best honey walnyt prawns that we have every tasted .
Predictions:
best            -> O
honey           -> B-POS
walnyt          -> I-POS
prawns          -> O
that            -> O
we              -> O
have            -> O
every           -> O
tasted          -> O
.               -> O

Sentence:
The food there are sastifying .
Predictions:
The             -> O
food            -> B-POS
there           -> O
are             -> O
sastifying      -> O
.               -> O

Sentence:
The one veget

{'all_preds': [['O',
   'O',
   'O',
   'B-POS',
   'O',
   'O',
   'B-POS',
   'I-POS',
   'I-POS',
   'B-POS',
   'I-POS',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O'],
  ['O', 'B-POS', 'I-POS', 'O', 'O', 'O', 'O', 'O', 'O', 'O'],
  ['O', 'B-POS', 'O', 'O', 'O', 'O'],
  ['O',
   'O',
   'B-POS',
   'I-POS',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'B-POS',
   'I-POS',
   'O',
   'O',
   'O',
   'B-POS',
   'I-POS',
   'O',
   'B-POS',
   'O'],
  ['O',
   'B-POS',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'I-POS',
   'I-POS',
   'O',
   'O',
   'O',
   'B-POS',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O'],
  ['O', 'B-POS', 'O', 'B-POS', 'I-POS', 'O', 'O', 'O', 'O'],
  ['B-NEU',
   'O',
   'O',
   'O',
   'B-NEG',
   'I-NEG',
   'O',
   'O',
   'O',
   'O',
   'B-NEG',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'O',
   'B-NEU',
   'O']